# Capstone, a lane for every job

**Scenario:** a claims team runs two workloads against one model. An adjuster waits on screen for a
first assessment, and overnight a queue re-scores ninety thousand open claims. Both were built the
same way, on the model with the lowest advertised price.

The adjuster waits. The overnight run is genuinely cheap. One decision was right and the same
decision was wrong, on the same day, in the same product.

Three lanes exist, and they are **post, courier and freight**. Same parcel, three answers, and the
question is never which is best.

## Mechanics

Three levers, and each pays in a different currency.

| Lever | Buys you | Costs you | Where the number comes from |
|---|---|---|---|
| Model choice | Money, or speed, rarely both | The other one | Measured cost and the clock |
| Work shape | Wall clock, `max` instead of `sum` | Nothing in money | The clock |
| Batch lane | About half the token price | Interactivity, entirely | The price list |

Money and time are **different currencies**, and a model is usually cheap in one and expensive in the
other. There is no single best model, only a best model for a deadline.

## The picture

![Two workloads, three lanes, one decision each](images/lanes.svg)

The same claim text goes down all three lanes. What differs is what you are prepared to wait for.

## The cost

```
interactive = prompt_tokens x (1 - cached_share) x rate + completion_tokens x rate
overnight   = the same, at the batch rate, if you can wait
```

Every term is measured, not quoted. `cached_share` comes from a warm call, and both rates come from
the probe.

## The failure

Pick the lowest advertised prompt price, run the work in a loop, and see what it costs.

In [1]:
import time
from vault import get_client, load_env, model_for, provider_truth
from vault.costs import Usage, cost_of

load_env()
client = get_client("13-cost-and-latency-at-volume/03-capstone-a-lane-for-every-job")

CLAIMS = ["water damage, kitchen ceiling, tenant absent",
          "rear-end collision, no injury, third party admits",
          "stolen bicycle, no forced entry, receipt provided",
          "storm damage, roof tiles, neighbour affected too"]

rates = provider_truth()["models"]
cheapest = min(rates, key=lambda m: float(rates[m]["prompt_usd_per_token"]))
print(f"lowest advertised prompt price: {cheapest}")

lowest advertised prompt price: mistralai/mistral-nemo


One assessment per claim, in a loop, on the model the price list favours.

In [2]:
def assess(model, claim):
    """One assessment. Returns what it really cost and how long it took."""
    started = time.monotonic()
    reply = client.chat.completions.create(
        model=model, max_tokens=120,
        messages=[{"role": "system", "content": "You triage insurance claims. Two sentences."},
                  {"role": "user", "content": claim}])
    return cost_of(Usage.from_response(reply)), time.monotonic() - started

Run the adjuster's workload the way it shipped, and hold it to the promise on screen on screen.

In [3]:
SCREEN_BUDGET_SECONDS = 3.0

started = time.monotonic()
naive = [assess(cheapest, c) for c in CLAIMS]
naive_wall = time.monotonic() - started
naive_cost = sum(cost for cost, _ in naive)

print(f"model     : {cheapest}")
print(f"cost      : ${naive_cost:.8f} for {len(CLAIMS)} claims")
print(f"wall clock: {naive_wall:.2f}s   budget {SCREEN_BUDGET_SECONDS:.1f}s")
assert naive_wall <= SCREEN_BUDGET_SECONDS, f"adjuster waited {naive_wall:.2f}s"

model     : mistralai/mistral-nemo
cost      : $0.00000438 for 4 claims
wall clock: 22.74s   budget 3.0s


AssertionError: adjuster waited 22.74s

## The diagnosis

The adjuster waited well past the budget. Here is the uncomfortable part: **the model choice was
correct on cost.** The cheapest per token really was the cheapest in total on this workload. Nothing
about the money was wrong.

It was optimised in the wrong currency. Look at the slowest column below: the same model that wins on
cost takes several seconds per claim, and cheap and slow travel together because the price reflects
what you are getting. Somebody chose once, for the product, when the product has two workloads with
opposite deadlines.

Two levers are left and they are not equal. Changing the **shape** costs nothing, because the same
tokens are read either way. Changing the **model** costs money on every call, forever. So the order
matters: pull the free one first, and pay only if it is not enough.

## The fix

Measure both currencies on the real workload, then choose per deadline rather than once for the
product.

In [4]:
def profile(model, claims):
    """What this model really costs and how long it really takes, on this work."""
    results = [assess(model, c) for c in claims]
    return {"model": model,
            "cost": sum(c for c, _ in results),
            "slowest": max(t for _, t in results)}

Compare the candidates on the same four claims, and read the two columns as separate answers.

In [5]:
candidates = [profile(model_for(r), CLAIMS) for r in ("default", "reasoning", "small")]
for p in sorted(candidates, key=lambda p: p["cost"]):
    print(f"  {p['model']:28} ${p['cost']:.8f}   slowest {p['slowest']:.2f}s")

cheapest_run = min(candidates, key=lambda p: p["cost"])
fastest_run = min(candidates, key=lambda p: p["slowest"])

print()
print(f"cheapest: {cheapest_run['model']} at {cheapest_run['slowest']:.2f}s per claim")
print(f"fastest : {fastest_run['model']} at "
      f"{fastest_run['cost'] / cheapest_run['cost']:.0f}x the cost")

  mistralai/mistral-nemo       $0.00000504   slowest 5.58s
  google/gemini-2.5-flash-lite $0.00006500   slowest 1.67s
  openai/gpt-5-nano            $0.00010805   slowest 2.48s

cheapest: mistralai/mistral-nemo at 5.58s per claim
fastest : google/gemini-2.5-flash-lite at 13x the cost


The free lever first. Same model, same tokens, same bill: overlap the calls.

In [6]:
from concurrent.futures import ThreadPoolExecutor


def run_parallel(model, claims):
    """Same claims, overlapped. Returns total cost and wall clock."""
    started = time.monotonic()
    with ThreadPoolExecutor(max_workers=len(claims)) as pool:
        out = list(pool.map(lambda c: assess(model, c), claims))
    return sum(cost for cost, _ in out), time.monotonic() - started

The slow model, overlapped, against the promise on screen.

In [7]:
_, cheap_parallel = run_parallel(cheapest_run["model"], CLAIMS)
print(f"cheap model, serial   : {naive_wall:.2f}s")
print(f"cheap model, parallel : {cheap_parallel:.2f}s")
print(f"budget                : {SCREEN_BUDGET_SECONDS:.1f}s")
print(f"inside budget         : {cheap_parallel <= SCREEN_BUDGET_SECONDS}")

cheap model, serial   : 22.74s
cheap model, parallel : 6.25s
budget                : 3.0s
inside budget         : False


A large improvement for no extra money, which is why the free lever goes first. Whether it is
**enough** depends on your budget, and the numbers above say whether it was here.

There is a second thing in that measurement worth more than the average. Run this a few times and
the cheap model's wall clock moves a long way, while the fast one barely does. A model whose latency
swings is a risk on a path where somebody is waiting, and an average hides that completely.

So price the second lever, and decide with a number rather than a preference.

In [8]:
screen_cost, screen_wall = run_parallel(fastest_run["model"], CLAIMS)
night_cost = cheapest_run["cost"]

print(f"cheap model, overlapped : ${night_cost:.8f}  {cheap_parallel:.2f}s")
print(f"fast model, overlapped  : ${screen_cost:.8f}  {screen_wall:.2f}s")
print()
print(f"the fast model costs {screen_cost / night_cost:.0f}x and buys "
      f"{cheap_parallel - screen_wall:.2f}s more headroom")
print("worth it only if the cheap lane cannot hold the budget under load")

cheap model, overlapped : $0.00000504  6.25s
fast model, overlapped  : $0.00007140  0.67s

the fast model costs 14x and buys 5.57s more headroom
worth it only if the cheap lane cannot hold the budget under load


The third lane is the overnight queue, and it is the one this notebook cannot demonstrate.

Batch models exist here under a `:batch` suffix and are listed at about half the price. They are
also refused by the chat endpoint, which is the API telling you the lane is a different road rather
than a flag. Submitting to it needs a file upload and a batch endpoint, and **that endpoint returned
404 on the account this course was recorded with**, so the flow below is shown and not run.

In [9]:
def batch_saving(model, tokens_per_claim=400, claims=90_000):
    """What the overnight queue would cost live. The batch lane is not measured."""
    live = float(rates[model]["prompt_usd_per_token"])
    print(f"  live lane : {model}")
    print(f"  batch lane: {model}:batch, refused by the chat endpoint by design")
    return tokens_per_claim * claims * live


print(f"overnight queue on the live lane: ${batch_saving(cheapest_run['model']):.2f}")
print("a batch lane at half price would halve that, if you can wait")

  live lane : mistralai/mistral-nemo
  batch lane: mistralai/mistral-nemo:batch, refused by the chat endpoint by design
overnight queue on the live lane: $0.68
a batch lane at half price would halve that, if you can wait


## The gate

The regression worth preventing is someone reaching for a dearer model when the shape was the
problem. This pins the order the levers are pulled in.

In [10]:
def test_shape_is_tried_before_money():
    """Cheap and fast are different models, and the free lever comes first."""
    assert cheapest_run["model"] != fastest_run["model"], (
        "one model won both currencies, so re-measure before splitting lanes")
    assert fastest_run["cost"] > cheapest_run["cost"], "the fast model was not dearer"
    assert cheap_parallel < naive_wall, (
        f"overlapping did not help: {cheap_parallel:.2f}s against {naive_wall:.2f}s")


test_shape_is_tried_before_money()
print("gate holds: cheap and fast differ, and the shape lever pays first")

gate holds: cheap and fast differ, and the shape lever pays first


### Enterprise exploration

- The cheap model's wall clock varies a lot between runs. What percentile do you promise, and how
  would you measure it rather than quoting an average?
- The overnight lane halves the token price and costs you a day. Which of your workloads could
  actually wait, and who signs off that they can?
- Fanning out five ways per claim is fine. What does the connection pool cost at ninety thousand
  claims, and where does it break first?
- A regulator asks how a claim was assessed. Does your answer change if the model changed last month
  for cost reasons, and is that recorded?

### Key takeaways

- Money and time are different currencies. A model is usually cheap in one and dear in the other.
- Shape is free and model is not, so change the shape first and price the model second.
- An average latency hides variance, and variance is what breaks a promise on screen.
- Choose per deadline, not per product. Two workloads want two answers.
- A lane you cannot reach is still worth knowing about, and worth saying you have not run.